# 🧠 Intersection over Union (IoU): Mathematical Overlap and Loss Metrics

Welcome to the hands-on explanation notebook for **Intersection over Union (IoU)**! In this notebook, we will:
1. Explain the theory, step-by-step derivation, and importance of IoU.
2. Implement IoU calculation from scratch in NumPy.
3. Define bounding boxes with varying degrees of overlap:
   - High Overlap.
   - Low Overlap.
   - Zero Overlap.
4. Calculate and compare IoUs for all cases.
5. Plot the bounding boxes side-by-side using Matplotlib to visually illustrate intersection regions.
6. Discuss advanced IoU loss formulations (GIoU, DIoU, CIoU) utilized in modern object detection (YOLO).

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Set seed for reproducibility
np.random.seed(42)

## 1. Scratch Implementation in NumPy

We write a function that takes two bounding boxes in XYXY format and computes the exact float overlap ratio.

In [ ]:
def calculate_iou(boxA, boxB):
    x1_inter = max(boxA[0], boxB[0])
    y1_inter = max(boxA[1], boxB[1])
    x2_inter = min(boxA[2], boxB[2])
    y2_inter = min(boxA[3], boxB[3])
    
    width_inter = max(0.0, x2_inter - x1_inter)
    height_inter = max(0.0, y2_inter - y1_inter)
    area_inter = width_inter * height_inter
    
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    area_union = areaA + areaB - area_inter
    
    if area_union == 0.0:
        return 0.0
        
    return area_inter / area_union

## 2. Setting Up Bounding Box Cases

We define four boxes in XYXY format:
-   **Reference Ground Truth Box A:** `[100, 100, 200, 200]`
-   **Box B (High Overlap):** `[120, 120, 220, 220]`
-   **Box C (Low Overlap):** `[180, 180, 280, 280]`
-   **Box D (Zero Overlap):** `[250, 250, 350, 350]`

In [ ]:
box_truth = [100.0, 100.0, 200.0, 200.0]
box_high = [120.0, 120.0, 220.0, 220.0]
box_low = [180.0, 180.0, 280.0, 280.0]
box_zero = [250.0, 250.0, 350.0, 350.0]

iou_high = calculate_iou(box_truth, box_high)
iou_low = calculate_iou(box_truth, box_low)
iou_zero = calculate_iou(box_truth, box_zero)

print(f"High Overlap IoU: {iou_high:.4f}")
print(f"Low Overlap IoU : {iou_low:.4f}")
print(f"Zero Overlap IoU: {iou_zero:.4f}")

## 3. Visualizing Box Overlaps

Let's plot these three cases side-by-side using Matplotlib to visualize how the intersection region changes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

boxes = [box_high, box_low, box_zero]
titles = [f"High Overlap (IoU = {iou_high:.4f})", f"Low Overlap (IoU = {iou_low:.4f})", f"Zero Overlap (IoU = {iou_zero:.4f})"]

for idx, ax in enumerate(axes):
    ax.set_xlim(50, 400)
    ax.set_ylim(400, 50)
    
    rect_truth = patches.Rectangle((box_truth[0], box_truth[1]), 100, 100, linewidth=3, edgecolor='green', facecolor='none', label='Ground Truth')
    ax.add_patch(rect_truth)
    
    p_box = boxes[idx]
    w = p_box[2] - p_box[0]
    h = p_box[3] - p_box[1]
    rect_pred = patches.Rectangle((p_box[0], p_box[1]), w, h, linewidth=3, edgecolor='red', facecolor='none', label='Prediction')
    ax.add_patch(rect_pred)
    
    x1_i = max(box_truth[0], p_box[0])
    y1_i = max(box_truth[1], p_box[1])
    x2_i = min(box_truth[2], p_box[2])
    y2_i = min(box_truth[3], p_box[3])
    
    w_i = max(0.0, x2_i - x1_i)
    h_i = max(0.0, y2_i - y1_i)
    if w_i > 0 and h_i > 0:
        rect_inter = patches.Rectangle((x1_i, y1_i), w_i, h_i, facecolor='blue', alpha=0.3, label='Intersection')
        ax.add_patch(rect_inter)
        
    ax.set_title(titles[idx], fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Why YOLO Uses CIoU Loss

Standard IoU has two main limitations:
1.  **Gradient Saturation:** When boxes do not overlap, $\text{IoU} = 0$. The gradient of the loss function is 0, so the network cannot learn how to shift the boxes closer.
2.  **Lack of Shape Alignment:** Raw IoU does not take into account how well the aspect ratios of the boxes align.

To solve this, YOLO uses **Complete IoU (CIoU) Loss**:
$$\mathcal{L}_{\text{CIoU}} = 1 - \text{IoU} + \frac{\rho^2(\mathbf{b}, \mathbf{b}^{gt})}{c^2} + \alpha v$$

Where:
-   $\rho(\mathbf{b}, \mathbf{b}^{gt})$ is the Euclidean distance between the center points of the prediction and ground truth boxes.
-   $c$ is the diagonal length of the smallest enclosing box containing both boxes.
-   $\alpha v$ is an aspect ratio consistency term that measures how well the prediction's width/height ratio matches the ground truth's ratio.
This ensures stable box updates even under zero overlap!